In [29]:
import numpy as np
import pandas as pd
import warnings
import graphviz
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
import seaborn as sns

In [30]:
class new_1_category_people():
    def __init__(self,df):
        self.df = pd.read_csv(df)


    def generate(self, N_new,target_column,target_value,categorical_cols,randomChoiceCategorialCols,bool_copy_categorial_col = True):

        df = self.df
        healthy = df[df[target_column] == target_value].copy()

        target_col = target_column
        numeric_cols = [c for c in healthy.columns 
                        if c not in categorical_cols + [target_col]]

        N_new = N_new
        new_people = []

        # Вычисляем стандартные отклонения для шума
        std_devs = healthy[numeric_cols].std()

        for _ in range(N_new):
            # Случайно выбираем исходного человека
            original = healthy.sample(n=1).iloc[0]
            
            # Добавляем шум (5% от стандартного отклонения)
            noise = np.random.normal(0, std_devs * 0.05, len(numeric_cols))
            
            person = {}
            for i, col in enumerate(numeric_cols):
                person[col] = max(0, original[col] + noise[i])  # не допускаем отрицательных
                person[col] = round(person[col], 2)
            
            # Категориальные признаки копируем с вероятностью 80% (небольшой шум)
            if categorical_cols and bool_copy_categorial_col:
                for i in categorical_cols:
                    if np.random.random() > 0.2:
                        person[i] = original[i]

                    else:
                        person[i] = np.random.choice(randomChoiceCategorialCols)
            else:
                pass
            
            person[target_column] = target_value
            new_people.append(person)
        
        self.df_new = pd.DataFrame(new_people)

        # self.df_new.to_csv('saved_data/synthetic_healthy_people.csv', index=False)
        self.df_new["name"] = range(96, 96 + len(self.df_new))

        print(f"✅ Сгенерировано {N_new} человек методом бутстрапа с шумом")
        print(self.df_new.head())
    def merge_new_df(self, merge_columns):
        df_new_1_category_people = pd.merge(
        self.df, 
        self.df_new, 
        how='outer', 
        on=merge_columns
        )
        df_new_1_category_people = df_new_1_category_people.drop("name",axis=1)
        return df_new_1_category_people

In [31]:
new_1_category_df_with_data_spliting = new_1_category_people("saved_data/mean_df_with_spliting.csv")
merge_columns = [
    # Целевая переменная
    "здоров(1)\болен(0)", 
    
    # Биохимия
    "ПОЛ в плазме", 
    "ПОЛ в мембр. Эритроцитов", 
    "Лактат", 
    "Глюкоза", 
    "Общий белок", 
    "Мочевина", 
    "Кортизол", 
    "Зонулин", 
    "Холестерин", 
    
    # Липидный профиль (уже без суффиксов _x, _y)
    "ЭХС", 
    "ТГ", 
    "НЭЖК", 
    "СХ", 
    "ФЛ", 
    "ЭХ",
    "НеЖК", 
    "ХЛ", 
    "СЖК",
    
    # Демография
    "Пол", 
    "Возраст"
]

new_1_category_df_with_data_spliting.generate(65,target_column = 'здоров(1)\болен(0)',target_value = 1,categorical_cols =  ['Пол'],randomChoiceCategorialCols = [1,2])
df_with_data_spliting_with_new_1_category = new_1_category_df_with_data_spliting.merge_new_df(merge_columns)
df_with_data_spliting_with_new_1_category
df_with_data_spliting_with_new_1_category.to_csv('saved_data/df_with_data_spliting_with_new_1_category.csv', index=False)

✅ Сгенерировано 65 человек методом бутстрапа с шумом
   ПОЛ в плазме  ПОЛ в мембр. Эритроцитов  Лактат  Глюкоза  Общий белок  \
0          1.19                      5.28   77.67     4.25         3.69   
1          1.59                      6.31   84.90     5.63         6.71   
2          1.62                      5.77   73.84     6.79         2.26   
3          2.00                      5.28   75.46     5.50         3.42   
4          1.98                      4.54   58.75     5.60         4.26   

   Мочевина  Кортизол  Зонулин  Холестерин    ЭХС  ...   НеЖК     ХЛ     ФЛ  \
0      1.59     20.52   288.65        0.18  16.17  ...  35.83  11.32  33.71   
1      1.54     20.33   196.63        0.27  13.25  ...  20.13  11.95  13.76   
2      1.60     21.55   186.25        0.24  25.02  ...  29.82  17.85  11.18   
3      1.57     30.13   234.52        0.28  13.98  ...  22.91  19.64  18.14   
4      1.56     25.60   241.22        0.62  14.10  ...  31.73  13.38  11.80   

     СЖК     ТГ     Э

In [32]:
new_1_category_df_WO_data_spliting = new_1_category_people("saved_data/mean_df_without_spliting_data.csv")
merge_columns = [
    # Целевая переменная
    "здоров(1)\болен(0)", 
    
    # Биохимия
    "ПОЛ в плазме", 
    "ПОЛ в мембр. Эритроцитов", 
    "Лактат", 
    "Глюкоза", 
    "Общий белок", 
    "Мочевина", 
    "Кортизол", 
    "Зонулин", 
    "Холестерин", 
    
    # Липидный профиль (уже без суффиксов _x, _y)
    "ЭХС", 
    "ТГ", 
    "НЭЖК", 
    "СХ", 
    "ФЛ", 
    "ЭХ",
    "НеЖК", 
    "ХЛ", 
    "СЖК",
    
    # Демография
    "Пол", 
    "Возраст"
]

new_1_category_df_WO_data_spliting.generate(65,target_column = 'здоров(1)\болен(0)',target_value = 1,categorical_cols =  ['Пол'],randomChoiceCategorialCols = [1,2])
df_WO_data_spliting_with_new_1_category = new_1_category_df_WO_data_spliting.merge_new_df(merge_columns)
df_WO_data_spliting_with_new_1_category.to_csv('saved_data/df_WO_data_spliting_with_new_1_category.csv', index=False)

✅ Сгенерировано 65 человек методом бутстрапа с шумом
   ПОЛ в плазме  ПОЛ в мембр. Эритроцитов  Лактат  Глюкоза  Общий белок  \
0          0.77                      5.40   73.94     6.74         3.28   
1          1.19                      5.31   77.90     4.89         4.15   
2          1.63                      5.74   74.06     6.71         2.35   
3          1.23                      5.32   77.51     4.26         3.52   
4          1.25                      5.29   77.65     4.23         3.59   

   Мочевина  Кортизол  Зонулин  Холестерин    ЭХС  ...   НеЖК     ХЛ     ФЛ  \
0      1.56     30.76   234.51        0.24  16.10  ...  31.77  18.96  34.24   
1      1.54     23.03   190.64        0.27  21.98  ...  27.79  11.49  13.33   
2      1.50     21.97   186.39        0.25  24.67  ...  30.51  17.62  10.70   
3      1.51     20.54   290.35        0.18  15.60  ...  35.59  11.21  33.00   
4      1.46     20.34   289.40        0.18  15.78  ...  35.77  11.45  33.83   

     СЖК     ТГ     С

In [33]:
dfdf = pd.read_csv("/home/ikuku/UDGU_project/main/EEG_files/data_for_EEG/combined_horizontal_data.csv")

new_1_category_EEG = new_1_category_people("/home/ikuku/UDGU_project/main/EEG_files/data_for_EEG/combined_horizontal_data.csv")

new_1_category_EEG.generate(98,target_column = 'здоров(1)/болен(0)',target_value = 1,categorical_cols =  ["Имя"],randomChoiceCategorialCols = [],bool_copy_categorial_col = False)
common_cols = list(set(new_1_category_EEG.df.columns) & set(new_1_category_EEG.df_new.columns))

df_with_new_1_category_EEG = new_1_category_EEG.merge_new_df(common_cols)
df_with_new_1_category_EEG.to_csv('saved_data/df_with_new_1_category_EEG.csv', index=False)
df_with_new_1_category_EEG

✅ Сгенерировано 98 человек методом бутстрапа с шумом
   D_1:2  D_1:3  D_1:4  D_1:5  D_1:6  D_1:7  D_1:8  D_1:9  D_1:10  D_1:11  \
0   0.84   0.86   0.85   0.96   0.93   0.82   0.64   0.75    0.82    0.76   
1   0.78   0.73   0.86   0.94   0.93   0.75   0.58   0.80    0.87    0.75   
2   0.83   0.84   0.74   0.87   0.91   0.71   0.63   0.58    0.70    0.63   
3   0.60   0.68   0.62   0.78   0.70   0.69   0.66   0.65    0.60    0.64   
4   0.64   0.60   0.90   0.96   0.94   0.56   0.63   0.89    0.96    0.62   

   ...  B2_17:20  B2_17:21  B2_18:19  B2_18:20  B2_18:21  B2_19:20  B2_19:21  \
0  ...      0.78      0.82      0.49      0.71      0.80      0.61      0.51   
1  ...      0.81      0.84      0.57      0.74      0.81      0.68      0.58   
2  ...      0.66      0.62      0.59      0.71      0.67      0.63      0.59   
3  ...      0.75      0.74      0.58      0.64      0.63      0.65      0.51   
4  ...      0.83      0.87      0.55      0.71      0.78      0.59      0.55   

   

,Имя,D_1:2,D_1:3,D_1:4,D_1:5,D_1:6,D_1:7,D_1:8,D_1:9,D_1:10,...,B2_17:19,B2_17:20,B2_17:21,B2_18:19,B2_18:20,B2_18:21,B2_19:20,B2_19:21,B2_20:21,здоров(1)/болен(0)
0,f11192810,0.612,0.643,0.707,0.808,0.655,0.643,0.575,0.734,0.753,...,0.569,0.919,0.944,0.598,0.831,0.862,0.593,0.561,0.907,0
1,f30696560,0.834,0.720,0.726,0.792,0.736,0.724,0.746,0.688,0.739,...,0.612,0.791,0.616,0.561,0.627,0.556,0.601,0.542,0.585,0
2,f3896710,0.835,0.665,0.755,0.832,0.747,0.661,0.568,0.736,0.705,...,0.587,0.701,0.694,0.511,0.499,0.501,0.752,0.596,0.748,0
3,f100220150,0.896,0.846,0.787,0.955,0.838,0.876,0.689,0.839,0.844,...,0.617,0.745,0.545,0.582,0.609,0.546,0.607,0.584,0.756,0
4,f11068280,0.558,0.560,0.793,0.887,0.829,0.483,0.545,0.817,0.810,...,0.521,0.930,0.952,0.494,0.780,0.836,0.559,0.549,0.926,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
279,f81312810,0.815,0.832,0.665,0.658,0.655,0.606,0.686,0.576,0.423,...,0.506,0.840,0.842,0.537,0.824,0.827,0.535,0.536,1.000,0
280,NaN,0.850,0.850,0.900,0.910,0.800,0.780,0.750,0.720,0.700,...,0.600,0.880,0.880,0.540,0.830,0.850,0.640,0.630,1.000,1
281,f14046090,0.749,0.750,0.885,0.869,0.853,0.745,0.766,0.722,0.745,...,0.583,0.908,0.909,0.563,0.789,0.792,0.569,0.570,0.999,0
282,NaN,0.850,0.860,0.890,0.910,0.800,0.790,0.750,0.710,0.700,...,0.600,0.880,0.880,0.540,0.830,0.850,0.630,0.630,1.010,1
